# 03 - 模型评测 (三层评测体系)

在微调前后分别运行完整的三层评测体系，量化模型能力提升。

- **Layer 1**: 通用能力基准 (MMLU-Pro, GPQA, HumanEval, MATH, BBH)
- **Layer 2**: TRIZ定制评测 (原理识别、矛盾解决、案例质量、ARIZ完整性)
- **Layer 3**: 工程性能基准 (吞吐量、延迟、内存)

## 3.1 加载模型

In [ ]:
import sys
sys.path.append('/home/meerkat/mongoose_ai')

import torch
from utils.training_utils import load_model_and_tokenizer
from config import BASE_MODEL, MODELS_DIR, RESULTS_DIR

# 确定模型路径 (基座模型或微调后模型)
# model_path = os.path.join(MODELS_DIR, BASE_MODEL.split('/')[-1])  # 基座模型
model_path = os.path.join(MODELS_DIR, BASE_MODEL.split('/')[-1])

print(f"评测模型: {model_path}")

# 加载模型 (4-bit量化以节省内存)
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model, tokenizer = load_model_and_tokenizer(
    model_name_or_path=model_path,
    quantization_config={
        'load_in_4bit': True,
        'bnb_4bit_quant_type': 'nf4',
        'bnb_4bit_compute_dtype': 'float16',
        'bnb_4bit_use_double_quant': True,
    },
    device_map='auto',
    trust_remote_code=True,
)

print("\n模型加载完成，开始评测...")

## 3.2 Layer 1: 通用能力基准 (可选)

**注意**: 运行完整的lm-eval评测需要较长时间(数小时)，建议在微调前后各运行一次以对比。

In [ ]:
# Layer 1: 通用能力评测
from utils.benchmark_utils import run_lm_evaluation

# 选择要评测的任务
tasks = ["mmlu_pro", "gpqa", "humaneval", "math", "bbh"]

print("开始通用能力评测 (Layer 1)...")
print("注意: 这可能需要数小时完成")

# 运行评测
general_results = run_lm_evaluation(
    model_path=model_path,
    tasks=tasks,
    output_dir=RESULTS_DIR,
    num_fewshot=5,
    batch_size=1,
)

print("\nLayer 1 评测完成!")

## 3.3 Layer 2: TRIZ定制评测

In [ ]:
# Layer 2: TRIZ定制评测
from utils.benchmark_utils import run_triz_evaluation

print("开始TRIZ定制评测 (Layer 2)...")

triz_results = run_triz_evaluation(
    model=model,
    tokenizer=tokenizer,
    output_dir=RESULTS_DIR,
)

print("\nTRIZ评测详细结果:")
print(f"  原理识别准确率: {triz_results['principle_accuracy']['accuracy']:.2%}")
print(f"  矛盾解决得分: {triz_results['contradiction_resolution']['average_score']:.2%}")
print(f"  案例生成覆盖率: {triz_results['case_quality']['average_coverage']:.2%}")
print(f"  ARIZ完整性: {triz_results['ariz_completeness']['completeness']:.2%}")
print(f"  综合得分: {triz_results['overall_score']:.2%}")

## 3.4 Layer 3: 工程性能基准

In [ ]:
# Layer 3: 性能评测
from utils.benchmark_utils import run_performance_benchmark

print("开始性能评测 (Layer 3)...")

perf_results = run_performance_benchmark(
    model=model,
    tokenizer=tokenizer,
    output_dir=RESULTS_DIR,
    max_tokens=512,
)

print("\n性能评测完成!")

## 3.5 生成综合评测报告

In [ ]:
# 聚合所有评测结果
from utils.benchmark_utils import aggregate_results

report = aggregate_results(
    general_results=None,  # 如果完成了Layer 1，传入结果
    triz_results=triz_results,
    perf_results=perf_results,
    output_dir=RESULTS_DIR,
)

print("\n综合评测报告生成完成!")
print(f"报告保存位置: {RESULTS_DIR}")

## 3.6 清理显存

In [ ]:
# 清理显存，为训练做准备
del model
del tokenizer
torch.cuda.empty_cache()

print("显存已清理")
print(f"当前显存占用: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

---

## 下一步

评测完成！记录基线分数后，请打开: **04_qlora_finetune.ipynb** 进行模型微调